# 데이터 탐색
1. XML 파일 로드
2. XML 구조 탐색
3. 주요 태그 확인
4. annotation 구조 이해
5. JSON label 확인
6. extract_nodule_info() 구현
7. 테스트

In [2]:
# =========================================================
# 데이터 경로 확인
# =========================================================
import os

PATH = "/home/hdo/lidc_project/data"
BASE_PATH = "/home/hdo/lidc_project/data/lidc-idri"

print(os.listdir(PATH))
print(os.listdir(BASE_PATH))

['lidc-idri', 'processed']
['LIDC-XML-only', 'manifest-1600709154662', 'slices_png', 'lidc-idri-nodule-counts-6-23-2015.xlsx', 'slices', 'tcia-diagnosis-data-2012-04-20.xlsx', 'nodule_malignancy_scores.json']


In [3]:
# =========================================================
# XML 파일 수집 
# - tcia-lidc-xml 경로 지정
# =========================================================
import glob

xml_files = sorted(glob.glob(
    os.path.join(
        BASE_PATH,
        "LIDC-XML-only/tcia-lidc-xml/**/*.xml"
    ),
    recursive=True
))

print("Number of XML files:", len(xml_files))
print(xml_files[:5])

Number of XML files: 1318
['/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/158.xml', '/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/159.xml', '/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/160.xml', '/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/161.xml', '/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/162.xml']


3. 전체 태그 구조 파악

In [4]:
# =========================================================
# root 확인
# =========================================================
import xml.etree.ElementTree as ET

tree = ET.parse(xml_files[0])
root = tree.getroot()

print(root.tag)

{http://www.nih.gov}LidcReadMessage


In [5]:
# root 아래 태그
print("\n===== ROOT CHILDREN =====")

for child in root:
    print(child.tag)


===== ROOT CHILDREN =====
{http://www.nih.gov}ResponseHeader
{http://www.nih.gov}readingSession
{http://www.nih.gov}readingSession
{http://www.nih.gov}readingSession
{http://www.nih.gov}readingSession


In [6]:
# child 아래 구조 확인
print("\n===== READING SESSION STRUCTURE =====")

for child in root:

    if "readingSession" in child.tag:

        for sub in child:
            print(sub.tag)

        break


===== READING SESSION STRUCTURE =====
{http://www.nih.gov}annotationVersion
{http://www.nih.gov}servicingRadiologistID
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}nonNodule
{http://www.nih.gov}nonNodule


In [7]:
# 결절 내부 구조 분석
print("\n===== FIRST NODULE DETAIL =====")

for elem in root.iter():

    if "unblindedReadNodule" in elem.tag:

        for child in elem:

            print("\nCHILD TAG:", child.tag)

            for sub in child.iter():

                print("   ", sub.tag, ":", sub.text)

        break


===== FIRST NODULE DETAIL =====

CHILD TAG: {http://www.nih.gov}noduleID
    {http://www.nih.gov}noduleID : 0

CHILD TAG: {http://www.nih.gov}roi
    {http://www.nih.gov}roi : 
    
    {http://www.nih.gov}imageZposition : 1604.5
    {http://www.nih.gov}imageSOP_UID : 1.3.6.1.4.1.14519.5.2.1.6279.6001.192028152416081898151427003898
    {http://www.nih.gov}inclusion : TRUE
    {http://www.nih.gov}edgeMap : None
    {http://www.nih.gov}xCoord : 177
    {http://www.nih.gov}yCoord : 353


실제 필요한 정보 가져오기

In [8]:
# =========================================================
# SeriesInstanceUid 찾기
# =========================================================

print("\n===== SERIES INSTANCE UID =====")

for elem in root.iter():

    if "SeriesInstanceUid" in elem.tag:

        print("Series UID:", elem.text)


# =========================================================
# malignancy 찾기
# =========================================================

print("\n===== MALIGNANCY =====")

for elem in root.iter():

    if "malignancy" in elem.tag:

        print("malignancy:", elem.text)


# =========================================================
# imageZposition 찾기
# =========================================================

print("\n===== IMAGE Z POSITION =====")

z_count = 0

for elem in root.iter():

    if "imageZposition" in elem.tag:

        print("z:", elem.text)

        z_count += 1

        if z_count > 10:
            break


===== SERIES INSTANCE UID =====
Series UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102183795724852353824

===== MALIGNANCY =====
malignancy: 3
malignancy: 3
malignancy: 4
malignancy: 5
malignancy: 4
malignancy: 4
malignancy: 5
malignancy: 1
malignancy: 5
malignancy: 5
malignancy: 3
malignancy: 3
malignancy: 3

===== IMAGE Z POSITION =====
z: 1604.5
z: 1550.5
z: 1547.5
z: 1544.5
z: 1541.5
z: 1538.5
z: 1535.5
z: 1487.5
z: 1484.5
z: 1481.5
z: 1478.5


In [9]:
# =========================================================
# edgeMap 좌표 확인
# =========================================================

print("\n===== EDGE MAP COORDINATES =====")

edge_count = 0

for elem in root.iter():

    if "edgeMap" in elem.tag:

        x = None
        y = None

        for child in elem:

            if "xCoord" in child.tag:
                x = child.text

            if "yCoord" in child.tag:
                y = child.text

        print(f"(x={x}, y={y})")

        edge_count += 1

        if edge_count > 20:
            break


===== EDGE MAP COORDINATES =====
(x=177, y=353)
(x=364, y=172)
(x=364, y=171)
(x=364, y=170)
(x=365, y=169)
(x=365, y=168)
(x=365, y=167)
(x=365, y=166)
(x=365, y=165)
(x=365, y=164)
(x=365, y=163)
(x=364, y=164)
(x=363, y=164)
(x=362, y=165)
(x=361, y=166)
(x=360, y=167)
(x=359, y=168)
(x=359, y=169)
(x=358, y=170)
(x=359, y=171)
(x=360, y=171)


## annotation 구조 확인

In [10]:
# =========================================================
# readingSession 개수 확인
# =========================================================

print("\n===== READING SESSION COUNT =====")

session_count = 0

for elem in root.iter():

    if "readingSession" in elem.tag:

        session_count += 1

print("Number of reading sessions:", session_count)


# =========================================================
# unblindedReadNodule 개수 확인
# =========================================================

print("\n===== NODULE COUNT =====")

nodule_count = 0

for elem in root.iter():

    if "unblindedReadNodule" in elem.tag:

        nodule_count += 1

print("Number of nodules:", nodule_count)



===== READING SESSION COUNT =====
Number of reading sessions: 4

===== NODULE COUNT =====
Number of nodules: 26


In [11]:
# 실제 UID 기준 데이터셋 개수 확인
uid_set = set()

for xml_path in xml_files:

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        for elem in root.iter():

            if "SeriesInstanceUid" in elem.tag:

                uid_set.add(elem.text)
                break

    except:
        continue

print("Unique CT Series:", len(uid_set))

Unique CT Series: 1018


하나씩 탐색했던 XML parsing 코드를
"실제 데이터 추출 함수" 형태로 정리
- Series UID 찾기
- malignancy 찾기
- edgeaMap 좌표 찾기
=> 하나의 함수로 통합

In [12]:
def extract_nodule_info(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    series_uid = None

    # -----------------------------------------------------
    # Series UID
    # -----------------------------------------------------

    for elem in root.iter():

        if "SeriesInstanceUid" in elem.tag:

            series_uid = elem.text
            break

    nodules = []

    # -----------------------------------------------------
    # Nodule Parsing
    # -----------------------------------------------------

    for nodule in root.iter():

        if "unblindedReadNodule" not in nodule.tag:
            continue

        malignancy = None
        coords = []

        # characteristic 내부 탐색
        for sub in nodule.iter():

            # malignancy
            if "malignancy" in sub.tag:
                malignancy = sub.text

            # edgeMap
            if "edgeMap" in sub.tag:

                x = None
                y = None

                for child in sub:

                    if "xCoord" in child.tag:
                        x = child.text

                    if "yCoord" in child.tag:
                        y = child.text

                coords.append((x, y))

        nodules.append({
            "series_uid": series_uid,
            "malignancy": malignancy,
            "coords": coords
        })

    return nodules

In [13]:
sample = extract_nodule_info(xml_files[0])

print("Number of nodules:", len(sample))

print(sample[0])

Number of nodules: 26
{'series_uid': '1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102183795724852353824', 'malignancy': None, 'coords': [('177', '353')]}
